# 🎨 DiffuSVG Pipeline v7 — SVG-T2I Paper Concepts

**Prompt → QLoRA fine-tuned Qwen2-VL → SVG**

Key additions over v6:
- **VFM Quality Gate** — DINOv2 cosine similarity filters bad training pairs (SVG-T2I §4.3)
- **Cross-Resolution Consistency** — catches blank/degenerate SVGs (SVG-T2I Fig 4)
- **CLIP + DINOv2 dual evaluation** — richer metrics than CLIP alone
- **Optional SVG-T2I backend** — high-res 1024×1024 reference images

---
⚠️ **Before you start:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run **Cell 1 (Install)** then `Runtime → Restart runtime`
3. Skip Cell 1 and run the rest top to bottom

In [ ]:
#@title ⚙️ Step 1 — Install Dependencies  (run once, then restart runtime)
import subprocess, sys

# Core training stack
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "peft>=0.13.0",
    "accelerate>=0.26.0",
    "transformers>=4.37.0",
])

# SVG rendering + evaluation
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "cairosvg",
    "open_clip_torch",
])

# Optional: SVG-T2I stage-0 reference generation
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "omegaconf>=2.1",
    "einops",
    "huggingface-hub>=0.20",
])

print("\n✅ All packages installed.")
print("👉 Now go to  Runtime → Restart runtime  then continue from Step 2.")

In [ ]:
#@title 🔍 Step 2 — Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU : {name}")
print(f"✅ VRAM: {total_gb:.1f} GB")
if total_gb < 14:
    print("⚠️  < 14 GB VRAM — the pipeline may OOM during training. Try reducing BATCH_SIZE.")

In [ ]:
#@title 📂 Step 3 — Upload Dataset  (training_pairs.json)
#
# Option A (recommended): Upload directly
# Option B: Mount Google Drive and point DATASET_PATH to the file

import os
from google.colab import files

USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_PATH = "/content/drive/MyDrive/DiffuSVG/training_pairs.json"  #@param {type:"string"}

DATASET_PATH = ""

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_PATH):
        DATASET_PATH = DRIVE_PATH
        print(f"✅ Dataset found at {DRIVE_PATH}")
    else:
        print(f"❌ File not found: {DRIVE_PATH}")
        print("   Check the path and re-run this cell.")
else:
    print("Uploading training_pairs.json ...")
    uploaded = files.upload()
    for fname in uploaded:
        dest = f"/content/{fname}"
        with open(dest, "wb") as f:
            f.write(uploaded[fname])
        DATASET_PATH = dest
        print(f"✅ Saved to {dest}")

print(f"\nDATASET_PATH = {DATASET_PATH!r}")

In [ ]:
#@title ⚙️ Step 4 — Configuration  { display-mode: "form" }

# ── VFM Quality Gate ──────────────────────────────────────────────────────────
VFM_MODEL             = "facebook/dinov2-small"  #@param ["facebook/dinov2-small", "facebook/dinov2-base"]
VFM_QUALITY_THRESHOLD = 0.35  #@param {type:"slider", min:0.1, max:0.8, step:0.05}
USE_VFM_FILTER        = True  #@param {type:"boolean"}

# ── VLM (Qwen2-VL) ───────────────────────────────────────────────────────────
VLM_MODEL   = "Qwen/Qwen2-VL-2B-Instruct"  #@param ["Qwen/Qwen2-VL-2B-Instruct"]
MAX_SEQ_LEN = 1024  #@param {type:"slider", min:512, max:2048, step:128}

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R       = 4   #@param {type:"slider", min:2, max:32, step:2}
LORA_ALPHA   = 16  #@param {type:"slider", min:4, max:64, step:4}
LORA_DROPOUT = 0.15  #@param {type:"slider", min:0.0, max:0.5, step:0.05}

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS        = 3  #@param {type:"slider", min:1, max:10, step:1}
BATCH_SIZE    = 1  #@param {type:"slider", min:1, max:4, step:1}
GRAD_ACCUM    = 8  #@param {type:"slider", min:1, max:32, step:1}
LEARNING_RATE = 1e-4  #@param ["5e-5", "1e-4", "2e-4"] {type:"raw"}

# ── SVG filter ───────────────────────────────────────────────────────────────
MAX_SVG_CHARS = 4000  #@param {type:"slider", min:500, max:10000, step:500}
MIN_SVG_CHARS = 50    #@param {type:"slider", min:10, max:500, step:10}

# ── Stage 0 (optional reference image generation via SVG-T2I / SD3.5) ────────
RUN_STAGE0   = False  #@param {type:"boolean"}
USE_SVGT2I   = True   #@param {type:"boolean"}

print("✅ Config saved.")

In [ ]:
#@title 🧠 Step 5 — Load Pipeline Code

import subprocess, sys, os, gc, json, logging, re, random, shutil, io
from pathlib import Path
from typing import List, Optional, Tuple, Dict

os.environ["CUDA_VISIBLE_DEVICES"]  = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_ALLOC_CONF"]   = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import numpy as np
from PIL import Image
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

WORKING_DIR = "/content"
os.makedirs(WORKING_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("DiffuSVG-v7")
log.info("Environment: colab | Working dir: /content")


# ── Apply config ──────────────────────────────────────────────────────────────
class Config:
    TRAINING_PAIRS_PATH: str        = DATASET_PATH
    MAX_SVG_CHARS: int              = MAX_SVG_CHARS
    MIN_SVG_CHARS: int              = MIN_SVG_CHARS
    VAL_SPLIT: float                = 0.1
    VFM_MODEL: str                  = VFM_MODEL
    VFM_QUALITY_THRESHOLD: float    = VFM_QUALITY_THRESHOLD
    VFM_CONSISTENCY_RESOLUTIONS: Tuple = (224, 448)
    REF_IMAGE_RESOLUTION: int       = 1024
    USE_SVGT2I: bool                = USE_SVGT2I
    VLM_MODEL: str                  = VLM_MODEL
    MAX_SEQ_LEN: int                = MAX_SEQ_LEN
    LORA_R: int                     = LORA_R
    LORA_ALPHA: int                 = LORA_ALPHA
    LORA_DROPOUT: float             = LORA_DROPOUT
    EPOCHS: int                     = EPOCHS
    BATCH_SIZE: int                 = BATCH_SIZE
    GRAD_ACCUM: int                 = GRAD_ACCUM
    LEARNING_RATE: float            = LEARNING_RATE
    WARMUP_RATIO: float             = 0.1
    OUTPUT_DIR: str                 = "/content/diffusvg_v7_output"
    LORA_OUTPUT_DIR: str            = "/content/diffusvg_v7_output/lora_checkpoints"
    EVAL_DIR: str                   = "/content/diffusvg_v7_output/evaluation"

cfg = Config()
for d in [cfg.OUTPUT_DIR, cfg.LORA_OUTPUT_DIR, cfg.EVAL_DIR]:
    os.makedirs(d, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════════
# SVG UTILITIES
# ════════════════════════════════════════════════════════════════════════════
_SVG_SYSTEM = """\
You are an SVG code generator. Given a text description, output ONLY the SVG \
element body (rect, circle, polygon, path, ellipse, line, etc.) that would appear \
inside <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">...</svg>.

Rules:
- Output ONLY SVG elements, no <svg> wrapper, no comments, no explanation.
- Always start with a background rect: <rect width="200" height="200" fill="#COLOR"/>
- Use solid fill colors (hex). No gradients, no filters, no blur.
- Keep it simple: aim for 3-25 elements maximum.
- Use geometric primitives: rect, circle, ellipse, polygon, line, path.
- All coordinates within 0-200 range.
"""

_FEW_SHOT_EXAMPLES = [
    ("a blue circle",
     '<rect width="200" height="200" fill="#ffffff"/>\n<circle cx="100" cy="100" r="60" fill="#1565C0"/>'),
    ("a red heart",
     '<rect width="200" height="200" fill="#ffffff"/>\n<circle cx="75" cy="85" r="30" fill="#E53935"/>\n<circle cx="125" cy="85" r="30" fill="#E53935"/>\n<polygon points="45,100 100,165 155,100" fill="#E53935"/>'),
    ("a house with red roof",
     '<rect width="200" height="200" fill="#E3F2FD"/>\n<rect x="50" y="110" width="100" height="80" fill="#FFF9C4"/>\n<polygon points="100,40 50,110 150,110" fill="#C62828"/>\n<rect x="88" y="150" width="25" height="40" fill="#5D4037"/>\n<rect x="60" y="125" width="20" height="20" fill="#81D4FA" stroke="#555" stroke-width="1"/>'),
    ("a rocket",
     '<rect width="200" height="200" fill="#0D1B2A"/>\n<polygon points="100,20 75,90 125,90" fill="#B0BEC5"/>\n<rect x="75" y="90" width="50" height="90" fill="#CFD8DC"/>\n<circle cx="100" cy="115" r="15" fill="#81D4FA"/>\n<polygon points="75,180 55,180 75,140" fill="#E53935"/>\n<polygon points="125,180 145,180 125,140" fill="#E53935"/>\n<polygon points="85,180 100,200 115,180" fill="#FF7043"/>'),
]

def _few_shot_block(prompt: str, n: int = 2) -> str:
    examples = random.sample(_FEW_SHOT_EXAMPLES, min(n, len(_FEW_SHOT_EXAMPLES)))
    parts = []
    for ex_prompt, ex_svg in examples:
        parts.append(f"Prompt: {ex_prompt}\nSVG:\n{ex_svg}\n")
    parts.append(f"Prompt: {prompt}\nSVG:")
    return "\n".join(parts)

def _wrap_svg(body: str) -> str:
    return f'<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n{body}\n</svg>'

def _render_svg_to_pil(svg_string: str, size: int = 200) -> Optional[Image.Image]:
    try:
        import cairosvg
        png_data = cairosvg.svg2png(bytestring=svg_string.encode("utf-8"),
                                    output_width=size, output_height=size)
        return Image.open(io.BytesIO(png_data)).convert("RGB")
    except Exception:
        return None

def _classify_complexity(svg: str) -> str:
    tags = re.findall(r"<(rect|circle|ellipse|polygon|polyline|line|path|text)\b", svg)
    n = len(tags)
    if n <= 3: return "simple"
    elif n <= 10: return "medium"
    return "complex"


# ════════════════════════════════════════════════════════════════════════════
# VFM QUALITY GATE  (SVG-T2I paper — Section 4.3, Figure 4)
# ════════════════════════════════════════════════════════════════════════════
class VFMQualityGate:
    def __init__(self, model_name: str = "facebook/dinov2-small", threshold: float = 0.35):
        self.model_name = model_name
        self.threshold  = threshold
        self._model     = None
        self._processor = None

    def _lazy_load(self):
        if self._model is not None:
            return
        from transformers import AutoImageProcessor, AutoModel
        log.info(f"[VFM] Loading {self.model_name}...")
        self._processor = AutoImageProcessor.from_pretrained(self.model_name)
        self._model     = AutoModel.from_pretrained(self.model_name)
        self._model.eval()
        if torch.cuda.is_available():
            self._model = self._model.cuda()
        log.info(f"[VFM] {self.model_name} loaded.")

    @torch.no_grad()
    def extract_features(self, image: Image.Image) -> torch.Tensor:
        self._lazy_load()
        inputs = self._processor(images=image, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        outputs = self._model(**inputs)
        feats   = outputs.last_hidden_state[:, 0, :]   # CLS token
        return feats / feats.norm(dim=-1, keepdim=True)

    def cosine_similarity(self, img_a: Image.Image, img_b: Image.Image) -> float:
        fa = self.extract_features(img_a)
        fb = self.extract_features(img_b)
        return float((fa @ fb.T).item())

    def cross_resolution_consistency(self, image: Image.Image,
                                     resolutions: Tuple = (224, 448)) -> Dict:
        self._lazy_load()
        res_list = list(resolutions)
        feats = {}
        for res in res_list:
            img_r = image.resize((res, res), Image.LANCZOS)
            feats[res] = self.extract_features(img_r)
        sims = {}
        for i in range(len(res_list) - 1):
            r1, r2 = res_list[i], res_list[i + 1]
            sims[f"{r1}\u2192{r2}"] = float((feats[r1] @ feats[r2].T).item())
        mean_c = float(np.mean(list(sims.values()))) if sims else 0.0
        return {"similarities": sims, "mean_consistency": mean_c,
                "is_consistent": mean_c >= 0.40}

    def gate_training_pair(self, rendered_svg: Image.Image,
                           reference_image: Optional[Image.Image] = None) -> Tuple[bool, Dict]:
        scores      = {}
        consistency = self.cross_resolution_consistency(
            rendered_svg, cfg.VFM_CONSISTENCY_RESOLUTIONS)
        scores["vfm_consistency"]    = consistency["mean_consistency"]
        scores["vfm_consistent"]     = consistency["is_consistent"]
        scores["vfm_resolution_sims"] = consistency["similarities"]
        if not consistency["is_consistent"]:
            return False, scores
        if reference_image is not None:
            sim = self.cosine_similarity(rendered_svg, reference_image)
            scores["vfm_svg_ref_sim"] = sim
            passes = sim >= self.threshold
        else:
            passes = True
        return passes, scores

    def unload(self):
        del self._model, self._processor
        self._model = self._processor = None
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        log.info("[VFM] Model unloaded.")


# ════════════════════════════════════════════════════════════════════════════
# STAGE 0 (OPTIONAL): REFERENCE IMAGE GENERATION
# ════════════════════════════════════════════════════════════════════════════
_SVGT2I_REPO = "https://github.com/KlingAIResearch/SVG-T2I.git"
_SVGT2I_HF   = "KlingTeam/SVG-T2I"

def _setup_svgt2i() -> Optional[str]:
    repo_root    = os.path.join(WORKING_DIR, "SVG-T2I")
    svg_t2i_dir  = os.path.join(repo_root, "svg_t2i")
    if not os.path.exists(repo_root):
        log.info(f"[SVG-T2I] Cloning {_SVGT2I_REPO} ...")
        r = subprocess.run(["git", "clone", "--depth", "1", _SVGT2I_REPO, repo_root],
                           capture_output=True, text=True)
        if r.returncode != 0:
            log.error(f"[SVG-T2I] git clone failed:\n{r.stderr[-600:]}")
            return None
    if not os.path.isdir(svg_t2i_dir):
        log.error(f"[SVG-T2I] Expected directory not found: {svg_t2i_dir}")
        return None
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "omegaconf>=2.1", "einops", "pytorch-lightning>=2.0",
                    "huggingface-hub>=0.20"], capture_output=True)
    pretrained_dir  = os.path.join(svg_t2i_dir, "pre-trained")
    autoencoder_cfg = os.path.join(pretrained_dir, "autoencoder",
                                   "svg_autoencoder_P_stage3_1024.yaml")
    dit_ckpt_dir    = os.path.join(pretrained_dir, "dit-stage4-T274M")
    if not os.path.exists(autoencoder_cfg) or not os.path.isdir(dit_ckpt_dir):
        log.info(f"[SVG-T2I] Downloading weights from {_SVGT2I_HF} ...")
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=_SVGT2I_HF, repo_type="model",
                              local_dir=pretrained_dir,
                              ignore_patterns=["*.md", "*.txt", "*.gitattributes"])
        except Exception as e:
            log.error(f"[SVG-T2I] Weight download failed: {e}")
            return None
    if not os.path.exists(autoencoder_cfg) or not os.path.isdir(dit_ckpt_dir):
        log.error("[SVG-T2I] Missing autoencoder or DiT checkpoint.")
        return None
    return svg_t2i_dir

def _generate_image_svgt2i(prompt: str, width=1024, height=1024) -> Optional[Image.Image]:
    svg_t2i_dir = _setup_svgt2i()
    if svg_t2i_dir is None: return None
    caption_path = os.path.join(WORKING_DIR, "svgt2i_caption.jsonl")
    with open(caption_path, "w") as f:
        f.write(json.dumps({"caption": prompt}) + "\n")
    out_dir = os.path.join(WORKING_DIR, "svgt2i_out")
    os.makedirs(out_dir, exist_ok=True)
    resolution      = max(width, height)
    pretrained_dir  = os.path.join(svg_t2i_dir, "pre-trained")
    autoencoder_cfg = os.path.join(pretrained_dir, "autoencoder", "svg_autoencoder_P_stage3_1024.yaml")
    dit_ckpt_dir    = os.path.join(pretrained_dir, "dit-stage4-T274M")
    cmd = [sys.executable, os.path.join(svg_t2i_dir, "sample_svg_t2i.py"),
           "--ckpt", dit_ckpt_dir, "--out_dir", out_dir,
           "--solver", "dpm", "--steps", "50",
           "--caption_path", caption_path, "--seed", "42",
           "--resolution", str(resolution), "--time_shifting_factor", "10",
           "--cfg_scale", "4.0", "--system_type", "base",
           "--autoencoder_config", autoencoder_cfg, "--batch_size", "1",
           "--rank", "0", "--ema"]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=svg_t2i_dir)
    if result.returncode != 0:
        log.warning(f"[SVG-T2I] Inference failed:\n{result.stderr[-600:]}")
        return None
    png_files = sorted(Path(out_dir).glob("*.png"), key=os.path.getmtime)
    if not png_files: return None
    img = Image.open(str(png_files[-1])).convert("RGB")
    if img.size != (width, height): img = img.resize((width, height), Image.LANCZOS)
    return img

def _generate_image_sd35(prompt: str, width=1024, height=1024) -> Optional[Image.Image]:
    try:
        from diffusers import StableDiffusion3Pipeline
        pipe = StableDiffusion3Pipeline.from_pretrained(
            "stabilityai/stable-diffusion-3.5-medium", torch_dtype=torch.float16)
        pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
        pipe.enable_model_cpu_offload()
        img = pipe(prompt=prompt, width=width, height=height,
                   num_inference_steps=28, guidance_scale=4.5).images[0]
        del pipe; gc.collect(); torch.cuda.empty_cache()
        return img
    except Exception as e:
        log.error(f"[SD3.5] Failed: {e}")
        return None

def generate_reference_image(prompt: str) -> Optional[Image.Image]:
    w = h = cfg.REF_IMAGE_RESOLUTION
    if cfg.USE_SVGT2I:
        img = _generate_image_svgt2i(prompt, w, h)
        if img is not None: return img
    return _generate_image_sd35(prompt, w, h)

def vectorize_to_svg(image: Image.Image, viewbox=200) -> Optional[str]:
    try:
        import tempfile
        img_gray = image.convert("L").resize((512, 512), Image.LANCZOS)
        with tempfile.NamedTemporaryFile(suffix=".bmp", delete=False) as bmp_f:
            img_gray.save(bmp_f.name)
            bmp_path = bmp_f.name
        svg_path = bmp_path.replace(".bmp", ".svg")
        subprocess.run(["potrace", "--svg", "--output", svg_path,
                        "--turdsize", "2", "--alphamax", "0.5", bmp_path],
                       check=True, capture_output=True)
        with open(svg_path) as f: raw_svg = f.read()
        os.unlink(bmp_path); os.unlink(svg_path)
        raw_svg = re.sub(r'width="[^"]*"',   f'width="{viewbox}"',         raw_svg)
        raw_svg = re.sub(r'height="[^"]*"',  f'height="{viewbox}"',        raw_svg)
        raw_svg = re.sub(r'viewBox="[^"]*"', f'viewBox="0 0 {viewbox} {viewbox}"', raw_svg)
        return raw_svg
    except Exception as e:
        log.warning(f"[Vectorize] {e}")
        return None

def build_training_pair_from_prompt(prompt: str) -> Optional[dict]:
    log.info(f"[Stage0] Generating for: {prompt}")
    ref_image = generate_reference_image(prompt)
    if ref_image is None: return None
    svg = vectorize_to_svg(ref_image)
    if svg is None: return None
    rendered = _render_svg_to_pil(svg, size=512)
    if rendered is None: return None
    ref_512 = ref_image.resize((512, 512), Image.LANCZOS)
    vfm = VFMQualityGate(cfg.VFM_MODEL, cfg.VFM_QUALITY_THRESHOLD)
    passes, scores = vfm.gate_training_pair(rendered, ref_512)
    vfm.unload()
    log.info(f"[Stage0] VFM scores: {scores} | passes={passes}")
    if not passes: return None
    return {"prompt": prompt, "svg": svg, "complexity": _classify_complexity(svg),
            "vfm_scores": scores, "source": "svgt2i_or_sd35+potrace"}


# ════════════════════════════════════════════════════════════════════════════
# STAGE 1: LOAD & VFM-FILTER DATASET
# ════════════════════════════════════════════════════════════════════════════
def _sort_by_curriculum(dataset: List[dict]) -> List[dict]:
    order = {"simple": 0, "medium": 1, "complex": 2}
    dataset.sort(key=lambda x: (order[x["complexity"]], x["svg_chars"]))
    return dataset

def load_and_filter_dataset(path: str, use_vfm_filter: bool = True) -> List[dict]:
    log.info(f"[Stage1] Loading dataset from {path}")
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    candidates = []
    for item in raw:
        svg    = item.get("svg", "")
        prompt = item.get("prompt", "")
        if not svg or not prompt: continue
        if len(svg) < cfg.MIN_SVG_CHARS or len(svg) > cfg.MAX_SVG_CHARS: continue
        candidates.append({"prompt": prompt, "svg": svg,
                            "complexity": _classify_complexity(svg),
                            "is_seed": item.get("is_seed", False),
                            "svg_chars": len(svg)})
    log.info(f"[Stage1] {len(candidates)} pass char-length filter (from {len(raw)} raw)")
    if not use_vfm_filter or len(candidates) == 0:
        return _sort_by_curriculum(candidates)

    log.info(f"[Stage1] Applying VFM cross-resolution filter ...")
    vfm = VFMQualityGate(cfg.VFM_MODEL, cfg.VFM_QUALITY_THRESHOLD)
    filtered, n_rej = [], 0
    for item in candidates:
        rendered = _render_svg_to_pil(item["svg"], size=max(cfg.VFM_CONSISTENCY_RESOLUTIONS))
        if rendered is None:
            n_rej += 1; continue
        consistency = vfm.cross_resolution_consistency(rendered, cfg.VFM_CONSISTENCY_RESOLUTIONS)
        item["vfm_consistency"]    = consistency["mean_consistency"]
        item["vfm_resolution_sims"] = consistency["similarities"]
        if consistency["is_consistent"]: filtered.append(item)
        else: n_rej += 1
    vfm.unload()
    log.info(f"[Stage1] VFM filter: kept {len(filtered)}, rejected {n_rej}")

    filtered.sort(key=lambda x: (
        {"simple": 0, "medium": 1, "complex": 2}[x["complexity"]],
        -x.get("vfm_consistency", 0)))
    stats = {}
    for d in filtered: stats[d["complexity"]] = stats.get(d["complexity"], 0) + 1
    log.info(f"[Stage1] Complexity breakdown: {stats}")
    return filtered


# ════════════════════════════════════════════════════════════════════════════
# STAGE 2: QLoRA FINE-TUNING
# ════════════════════════════════════════════════════════════════════════════
def build_chat_pair(prompt: str, svg_body: str, tokenizer) -> str:
    messages = [
        {"role": "system",    "content": _SVG_SYSTEM},
        {"role": "user",      "content": _few_shot_block(prompt, n=2)},
        {"role": "assistant", "content": svg_body},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

class SVGCausalDataset(torch.utils.data.Dataset):
    def __init__(self, data: list, tokenizer, max_len: int):
        self.samples = []
        skipped = 0
        for item in data:
            full_text  = build_chat_pair(item["prompt"], item["svg"], tokenizer)
            toks       = tokenizer(full_text, truncation=True, max_length=max_len,
                                   padding="max_length", return_tensors="pt")
            input_ids  = toks["input_ids"].squeeze()
            attn_mask  = toks["attention_mask"].squeeze()
            prompt_msgs = [
                {"role": "system", "content": _SVG_SYSTEM},
                {"role": "user",   "content": _few_shot_block(item["prompt"], n=2)},
            ]
            prompt_only = tokenizer.apply_chat_template(
                prompt_msgs, tokenize=False, add_generation_prompt=True)
            prompt_len  = len(tokenizer(prompt_only, truncation=True,
                                        max_length=max_len)["input_ids"])
            labels = input_ids.clone()
            labels[:prompt_len]     = -100
            labels[attn_mask == 0]  = -100
            if (labels != -100).sum() < 20:
                skipped += 1; continue
            self.samples.append({"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels})
        log.info(f"  SVGCausalDataset: {len(self.samples)} usable, {skipped} skipped")

    def __len__(self):         return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def train_lora(dataset: list):
    from transformers import (AutoTokenizer, Qwen2VLForConditionalGeneration,
                               BitsAndBytesConfig, TrainingArguments, Trainer)
    log.info("=" * 70)
    log.info("STAGE 2: QLoRA Fine-Tuning")
    log.info("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(cfg.VLM_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_gb  = torch.cuda.mem_get_info()[0] / 1e9
        total_gb = torch.cuda.mem_get_info()[1] / 1e9
        log.info(f"GPU: {free_gb:.1f} GB free / {total_gb:.1f} GB total")
        if free_gb < 1.0:
            log.error("< 1 GB GPU free — restart the runtime.")
            return None, None

    log.info(f"Loading {cfg.VLM_MODEL} with 4-bit NF4 quantization...")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        cfg.VLM_MODEL, quantization_config=quant_config,
        device_map={"":0}, trust_remote_code=True)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=cfg.LORA_R, lora_alpha=cfg.LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_dropout=cfg.LORA_DROPOUT, task_type=TaskType.CAUSAL_LM, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.is_parallelizable = model.model_parallel = False
    model.print_trainable_parameters()

    random.shuffle(dataset)
    split      = int(len(dataset) * (1 - cfg.VAL_SPLIT))
    train_data = dataset[:split]
    val_data   = dataset[split:]

    train_ds = SVGCausalDataset(train_data, tokenizer, cfg.MAX_SEQ_LEN)
    val_ds   = SVGCausalDataset(val_data,   tokenizer, cfg.MAX_SEQ_LEN) if val_data else None

    if len(train_ds) == 0:
        log.error("No usable training samples!")
        return None, None

    log.info(f"Train: {len(train_ds)}, Val: {len(val_ds) if val_ds else 0}")

    training_args = TrainingArguments(
        output_dir=cfg.LORA_OUTPUT_DIR,
        per_device_train_batch_size=cfg.BATCH_SIZE,
        per_device_eval_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=cfg.GRAD_ACCUM,
        num_train_epochs=cfg.EPOCHS,
        learning_rate=cfg.LEARNING_RATE,
        warmup_steps=max(1, int(cfg.WARMUP_RATIO
                                * (len(train_ds) // (cfg.BATCH_SIZE * cfg.GRAD_ACCUM))
                                * cfg.EPOCHS)),
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=5,
        eval_strategy="epoch" if val_ds and len(val_ds) > 0 else "no",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=bool(val_ds and len(val_ds) > 0),
        metric_for_best_model="eval_loss" if val_ds and len(val_ds) > 0 else None,
        report_to="none",
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
    )

    trainer = Trainer(model=model, args=training_args,
                      train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()

    adapter_dir = os.path.join(cfg.LORA_OUTPUT_DIR, "final_adapter")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    log.info(f"Adapter saved → {adapter_dir}")
    return model, tokenizer


# ════════════════════════════════════════════════════════════════════════════
# STAGE 3: INFERENCE
# ════════════════════════════════════════════════════════════════════════════
@torch.inference_mode()
def generate_svg(prompt: str, model, tokenizer, max_new_tokens: int = 1500) -> str:
    messages = [
        {"role": "system", "content": _SVG_SYSTEM},
        {"role": "user",   "content": _few_shot_block(prompt, n=2)},
    ]
    text    = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = tokenizer(text, return_tensors="pt").to(model.device)
    out     = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    svg_body = response.strip()
    svg_body = re.sub(r"^```(?:svg|xml|html)?\s*\n?", "", svg_body)
    svg_body = re.sub(r"\n?```\s*$", "", svg_body)
    if "<svg" in svg_body:
        m = re.search(r"<svg[^>]*>(.*?)</svg>", svg_body, re.DOTALL)
        if m: svg_body = m.group(1).strip()
    return _wrap_svg(svg_body)


# ════════════════════════════════════════════════════════════════════════════
# STAGE 4: VFM-ENHANCED EVALUATION
# ════════════════════════════════════════════════════════════════════════════
_TEST_PROMPTS = [
    "a purple butterfly", "a green leaf", "a blue diamond",
    "a red car", "an orange cat", "a yellow flower",
    "a pink umbrella", "a brown dog", "a gray cloud",
    "a white snowflake on blue background", "a gold trophy", "a silver key",
    "a black chess piece", "a rainbow flag", "a green cactus",
    "a blue whale", "a red fire truck", "an ice cream cone",
    "a smiling sun", "a crescent moon with stars",
]

def _load_clip():
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model     = model.float().eval()
    if torch.cuda.is_available(): model = model.cuda()
    return model, preprocess, tokenizer

@torch.no_grad()
def _clip_score(image, prompt, model, preprocess, tokenizer) -> float:
    import open_clip
    img_t = preprocess(image).unsqueeze(0)
    txt_t = tokenizer([prompt])
    if torch.cuda.is_available():
        img_t = img_t.cuda(); txt_t = txt_t.cuda()
    img_f = model.encode_image(img_t)
    txt_f = model.encode_text(txt_t)
    img_f /= img_f.norm(dim=-1, keepdim=True)
    txt_f /= txt_f.norm(dim=-1, keepdim=True)
    return float((img_f @ txt_f.T).item() * 100)

def evaluate_pipeline(model, tokenizer) -> dict:
    log.info("=" * 70)
    log.info("STAGE 4: Evaluation — CLIP + DINOv2 + Cross-Resolution Consistency")
    log.info("=" * 70)
    clip_model, clip_preprocess, clip_tok = _load_clip()
    vfm = VFMQualityGate(cfg.VFM_MODEL, cfg.VFM_QUALITY_THRESHOLD)
    Path(cfg.EVAL_DIR).mkdir(parents=True, exist_ok=True)
    results = []
    for i, prompt in enumerate(_TEST_PROMPTS):
        try:
            svg          = generate_svg(prompt, model, tokenizer)
            rendered_224 = _render_svg_to_pil(svg, size=224)
            rendered_448 = _render_svg_to_pil(svg, size=448)
            if rendered_224 is None:
                results.append({"prompt": prompt, "clip": 0.0, "dino_consistency": 0.0,
                                 "vfm_clip_product": 0.0, "success": False})
                continue
            rendered_224.save(os.path.join(cfg.EVAL_DIR, f"eval_{i:03d}.png"))
            with open(os.path.join(cfg.EVAL_DIR, f"eval_{i:03d}.svg"), "w") as f:
                f.write(svg)
            clip = _clip_score(rendered_224, prompt, clip_model, clip_preprocess, clip_tok)
            if rendered_448 is not None:
                consistency = vfm.cross_resolution_consistency(
                    rendered_448, cfg.VFM_CONSISTENCY_RESOLUTIONS)
            else:
                consistency = {"mean_consistency": 0.0, "similarities": {}, "is_consistent": False}
            dino_c = consistency["mean_consistency"]
            results.append({
                "prompt": prompt,
                "clip": round(clip, 2),
                "dino_consistency": round(dino_c, 4),
                "vfm_resolution_sims": consistency["similarities"],
                "vfm_clip_product": round(clip * dino_c, 2),
                "success": True,
            })
            log.info(f"  [{i+1:2d}/{len(_TEST_PROMPTS)}] "
                     f"CLIP={clip:.2f}  DINO={dino_c:.3f}  '{prompt[:40]}'")
        except Exception as e:
            log.error(f"  Eval error [{prompt}]: {e}")
            results.append({"prompt": prompt, "clip": 0.0, "dino_consistency": 0.0,
                             "vfm_clip_product": 0.0, "success": False})

    del clip_model; vfm.unload(); gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    successful = [r for r in results if r["success"]]
    if successful:
        clips = [r["clip"] for r in successful]
        dinos = [r["dino_consistency"] for r in successful]
        prods = [r["vfm_clip_product"] for r in successful]
        summary = {
            "n_total": len(results), "n_success": len(successful),
            "clip_mean":   round(float(np.mean(clips)),   2),
            "clip_median": round(float(np.median(clips)), 2),
            "clip_std":    round(float(np.std(clips)),    2),
            "dino_consistency_mean":   round(float(np.mean(dinos)),   4),
            "dino_consistency_median": round(float(np.median(dinos)), 4),
            "vfm_clip_product_mean":   round(float(np.mean(prods)),   2),
            "results": results,
        }
        log.info(f"  CLIP: mean={summary['clip_mean']:.2f}, median={summary['clip_median']:.2f}")
        log.info(f"  DINO consistency mean: {summary['dino_consistency_mean']:.3f}")
        log.info(f"  VFM×CLIP product mean: {summary['vfm_clip_product_mean']:.2f}")
    else:
        summary = {"n_total": len(results), "n_success": 0, "results": results}

    eval_path = os.path.join(cfg.EVAL_DIR, "eval_summary.json")
    with open(eval_path, "w") as f:
        json.dump(summary, f, indent=2)
    log.info(f"  Evaluation saved → {eval_path}")
    return summary


# ════════════════════════════════════════════════════════════════════════════
# HTML GALLERY
# ════════════════════════════════════════════════════════════════════════════
def generate_gallery(eval_dir: str):
    html = [
        '<!DOCTYPE html><html><head><meta charset="utf-8">',
        '<title>DiffuSVG v7 \u2014 VFM-Enhanced Evaluation</title>',
        '<style>',
        'body{background:#1a1a2e;color:#eee;font-family:Inter,sans-serif;padding:20px}',
        'h1{text-align:center;color:#e94560}',
        '.subtitle{text-align:center;color:#a8d8ea;margin-bottom:24px;font-size:13px}',
        '.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:20px;padding:20px}',
        '.card{background:#16213e;border-radius:12px;padding:15px;text-align:center;box-shadow:0 4px 15px rgba(0,0,0,.3)}',
        '.card img{width:200px;height:200px;border-radius:8px;background:#fff}',
        '.card .prompt{font-size:13px;margin:10px 0 5px;color:#a8d8ea}',
        '.card .clip{font-size:18px;font-weight:bold;color:#e94560}',
        '.card .dino{font-size:12px;color:#b8e0d2;margin-top:4px}',
        '.card .product{font-size:12px;color:#f9c74f;margin-top:2px}',
        '</style></head><body>',
        '<h1>DiffuSVG v7 \u2014 SVG-T2I Paper Concepts Applied</h1>',
        '<div class="subtitle">CLIP (text alignment) | DINO consistency (cross-resolution) | CLIP\u00d7DINO product</div>',
        '<div class="grid">',
    ]
    eval_json = os.path.join(eval_dir, "eval_summary.json")
    if os.path.exists(eval_json):
        with open(eval_json) as f: data = json.load(f)
        for i, r in enumerate(data.get("results", [])):
            img_path = f"eval_{i:03d}.png"
            clip_str = f"{r['clip']:.1f}" if r.get("success") else "FAIL"
            dino_str = f"DINO: {r.get('dino_consistency', 0):.3f}" if r.get("success") else ""
            prod_str = f"VFM\u00d7CLIP: {r.get('vfm_clip_product', 0):.1f}" if r.get("success") else ""
            html += [
                '<div class="card">',
                f'<img src="{img_path}" alt="{r[\"prompt\"]}">',
                f'<div class="prompt">{r["prompt"]}</div>',
                f'<div class="clip">CLIP: {clip_str}</div>',
                f'<div class="dino">{dino_str}</div>',
                f'<div class="product">{prod_str}</div>',
                '</div>',
            ]
    html += ['</div></body></html>']
    gallery_path = os.path.join(eval_dir, "gallery.html")
    with open(gallery_path, "w") as f: f.write("\n".join(html))
    log.info(f"Gallery saved \u2192 {gallery_path}")


print("\n✅ All pipeline functions loaded. Ready to run.")

In [ ]:
#@title 🚀 Step 6 — Run the Full Pipeline
#
# Stages:
#   0 (optional)  Generate new training pairs via SVG-T2I / SD3.5
#   1             Load & VFM-filter training_pairs.json
#   2             QLoRA fine-tune Qwen2-VL-2B
#   3+4           Inference on 20 test prompts + CLIP/DINOv2 evaluation

import logging

# ── Stage 0 (optional) ────────────────────────────────────────────────────
if RUN_STAGE0:
    stage0_prompts_path = "/content/stage0_prompts.txt"
    if os.path.exists(stage0_prompts_path):
        log.info("STAGE 0: Generating new training pairs via SVG-T2I / SD3.5")
        with open(stage0_prompts_path) as f:
            stage0_prompts = [l.strip() for l in f if l.strip()]
        new_pairs = []
        for p in stage0_prompts:
            pair = build_training_pair_from_prompt(p)
            if pair: new_pairs.append(pair)
        if new_pairs:
            stage0_out = os.path.join(cfg.OUTPUT_DIR, "stage0_pairs.json")
            with open(stage0_out, "w") as f: json.dump(new_pairs, f, indent=2)
            log.info(f"[Stage0] Generated {len(new_pairs)} pairs → {stage0_out}")
            if cfg.TRAINING_PAIRS_PATH and os.path.exists(cfg.TRAINING_PAIRS_PATH):
                with open(cfg.TRAINING_PAIRS_PATH) as f: existing = json.load(f)
                merged      = existing + new_pairs
                merged_path = os.path.join(cfg.OUTPUT_DIR, "training_pairs_merged.json")
                with open(merged_path, "w") as f: json.dump(merged, f, indent=2)
                cfg.TRAINING_PAIRS_PATH = merged_path
            else:
                cfg.TRAINING_PAIRS_PATH = stage0_out
    else:
        log.info("[Stage0] stage0_prompts.txt not found at /content/stage0_prompts.txt — skipping.")
        log.info("  Upload a file with one prompt per line to /content/stage0_prompts.txt to use Stage 0.")
else:
    log.info("[Stage0] Skipped (RUN_STAGE0=False).")

# ── Stage 1 ────────────────────────────────────────────────────────────────
if not cfg.TRAINING_PAIRS_PATH or not os.path.exists(cfg.TRAINING_PAIRS_PATH):
    raise FileNotFoundError(
        f"training_pairs.json not found at: {cfg.TRAINING_PAIRS_PATH!r}\n"
        "Please upload the file in Step 3 above.")

dataset = load_and_filter_dataset(cfg.TRAINING_PAIRS_PATH, use_vfm_filter=USE_VFM_FILTER)
if len(dataset) < 5:
    raise RuntimeError(f"Only {len(dataset)} samples after filtering — need ≥5.")

dataset_path = os.path.join(cfg.OUTPUT_DIR, "processed_dataset.json")
with open(dataset_path, "w") as f: json.dump(dataset, f, indent=2)
log.info(f"Processed dataset → {dataset_path} ({len(dataset)} pairs)")
for item in dataset[:5]:
    vfm_c = item.get("vfm_consistency", "N/A")
    log.info(f"  [{item['complexity']:7s}] chars={item['svg_chars']:5d}  "
             f"vfm_c={vfm_c if isinstance(vfm_c, str) else f'{vfm_c:.3f}'}  "
             f"'{item['prompt'][:50]}'")

# ── Stage 2 ────────────────────────────────────────────────────────────────
model, tokenizer = train_lora(dataset)
if model is None:
    raise RuntimeError("Training failed. Check logs above.")

# ── Stages 3 + 4 ───────────────────────────────────────────────────────────
eval_summary = evaluate_pipeline(model, tokenizer)
generate_gallery(cfg.EVAL_DIR)

# ── Zip outputs ─────────────────────────────────────────────────────────────
zip_base = "/content/diffusvg_v7_output"
shutil.make_archive(zip_base, "zip", cfg.OUTPUT_DIR)
log.info(f"Zipped → {zip_base}.zip")

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  PIPELINE COMPLETE")
print("="*60)
print(f"  Dataset:    {len(dataset)} pairs (VFM-filtered)")
print(f"  Adapter:    {cfg.LORA_OUTPUT_DIR}/final_adapter")
if eval_summary.get("n_success", 0) > 0:
    print(f"  CLIP mean:              {eval_summary['clip_mean']:.2f}")
    print(f"  DINO consistency mean:  {eval_summary['dino_consistency_mean']:.3f}  (paper: 0.60-0.90 for high-quality)")
    print(f"  VFM×CLIP product mean:  {eval_summary['vfm_clip_product_mean']:.2f}")
print(f"  Output zip: {zip_base}.zip")

In [ ]:
#@title 📊 Step 7 — Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

if eval_summary.get("n_success", 0) == 0:
    print("No successful evaluations to show.")
else:
    results = eval_summary["results"]
    successful = [r for r in results if r["success"]]

    # Score distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    clips = [r["clip"] for r in successful]
    dinos = [r["dino_consistency"] for r in successful]
    prods = [r["vfm_clip_product"] for r in successful]

    axes[0].hist(clips, bins=10, color='steelblue', edgecolor='black')
    axes[0].axvline(eval_summary['clip_mean'], color='red', linestyle='--',
                    label=f"mean={eval_summary['clip_mean']:.2f}")
    axes[0].set_title('CLIP Score Distribution'); axes[0].legend()

    axes[1].hist(dinos, bins=10, color='seagreen', edgecolor='black')
    axes[1].axvline(eval_summary['dino_consistency_mean'], color='red', linestyle='--',
                    label=f"mean={eval_summary['dino_consistency_mean']:.3f}")
    axes[1].set_title('DINO Consistency (paper: 0.60-0.90)'); axes[1].legend()

    axes[2].scatter(clips, dinos, alpha=0.7, color='purple')
    axes[2].set_xlabel('CLIP Score'); axes[2].set_ylabel('DINO Consistency')
    axes[2].set_title('CLIP vs DINO Correlation')

    plt.tight_layout()
    plt.savefig('/content/diffusvg_v7_output/score_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved score_distributions.png")

    # Top-5 SVG gallery
    top5 = sorted(successful, key=lambda r: r["vfm_clip_product"], reverse=True)[:5]
    indices = [results.index(r) for r in top5]
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, r, idx in zip(axes, top5, indices):
        img_path = os.path.join(cfg.EVAL_DIR, f"eval_{idx:03d}.png")
        if os.path.exists(img_path):
            ax.imshow(mpimg.imread(img_path))
        ax.set_title(f"{r['prompt'][:18]}\nCLIP={r['clip']:.1f}\nDINO={r['dino_consistency']:.3f}",
                     fontsize=8)
        ax.axis('off')
    plt.suptitle('Top-5 SVGs by VFM×CLIP Product', fontsize=12)
    plt.tight_layout()
    plt.savefig('/content/diffusvg_v7_output/top5_gallery.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved top5_gallery.png")

In [ ]:
#@title 💾 Step 8 — Download Outputs
from google.colab import files

zip_path = "/content/diffusvg_v7_output.zip"
if os.path.exists(zip_path):
    print(f"Downloading {zip_path} ...")
    files.download(zip_path)
    print("\nContents of the zip:")
    print("  lora_checkpoints/final_adapter/  — LoRA adapter weights + tokenizer")
    print("  evaluation/eval_*.png            — rendered SVG images")
    print("  evaluation/eval_*.svg            — raw SVG files")
    print("  evaluation/eval_summary.json     — CLIP + DINO scores")
    print("  evaluation/gallery.html          — visual gallery")
    print("  processed_dataset.json           — VFM-filtered training data")
else:
    print("Zip not found — did the pipeline complete? Check Step 6 for errors.")